In [2]:
from pathlib import Path
import json
import csv
from collections import Counter
from collections import defaultdict
import pandas as pd


file_path = Path("../../data/raw/entities.ftm.json")

schema_counts = Counter()

with file_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)
        schema_counts[obj.get("schema")] += 1

        if i >= 9999:   # first 10,000 lines
            break

print(schema_counts.most_common(20))

[('Occupancy', 6691), ('Person', 1005), ('Succession', 734), ('Company', 650), ('Organization', 329), ('LegalEntity', 181), ('Vessel', 114), ('Family', 94), ('Ownership', 83), ('Address', 48), ('Directorship', 41), ('Position', 12), ('UnknownLink', 10), ('Airplane', 8)]


In [ ]:
# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
input_file = Path("../../data/raw/entities.ftm.json")
output_dir = Path("../../data/processed/schema_csvs")
output_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def safe_value(value):
    """
    Preserve structure safely for CSV:
    - lists/dicts -> JSON string
    - everything else stays as is
    """
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False) # preserve structure as string
    return value # keep simple values unchanged


def flatten_entity(entity):
    """
    Flatten one entity safely:
    - top-level keys stay as they are
    - properties keys are prefixed with 'properties.'
    """
    flat = {} # this will store the flattened row
    
    # Loop over all top-level keys in the entity
    for key, value in entity.items():
        if key != "properties":  # skip properties for now
            flat[key] = safe_value(value)

    # Now handle nested 'properties'
    for key, value in entity.get("properties", {}).items():
        flat[f"properties.{key}"] = safe_value(value)

    return flat

# -------------------------------------------------------------------
# FIRST PASS:
# 1. collect rows by schema
# 2. collect all columns seen for each schema
# -------------------------------------------------------------------

# Dictionary:
# key = schema (e.g., Person, Company)
# value = list of rows belonging to that schema
rows_by_schema = defaultdict(list)

# Dictionary:
# key = schema
# value = set of column names seen for that schema
columns_by_schema = defaultdict(set)

# Open the JSON file
with input_file.open("r", encoding="utf-8") as f:
    
    # Read file line-by-line (each line = one entity)
    for line_num, line in enumerate(f, start=1):
        line = line.strip() # remove whitespace/newlines
        if not line:
            continue

        try:
            entity = json.loads(line)  # Convert JSON string → Python dictionary
        except json.JSONDecodeError as e:
            print(f"Skipping bad JSON on line {line_num}: {e}")
            continue

        schema = entity.get("schema", "UNKNOWN")
        flat = flatten_entity(entity)

        rows_by_schema[schema].append(flat)
        columns_by_schema[schema].update(flat.keys())

# -------------------------------------------------------------------
# SECOND PASS:
# write one CSV per schema
# -------------------------------------------------------------------

# Loop through each schema group
for schema, rows in rows_by_schema.items():
    
    # Get all columns seen for this schema (sorted for consistency)
    columns = sorted(columns_by_schema[schema])
    
    # Define output file name (e.g., Person.csv)
    output_file = output_dir / f"{schema}.csv"

    # Open CSV file for writing
    with output_file.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()

        for row in rows:
            # Ensure every column exists in the row
            # If missing → fill with empty string
            complete_row = {col: row.get(col, "") for col in columns}
            writer.writerow(complete_row)
    
    print(f"Saved: {output_file} ({len(rows)} rows, {len(columns)} columns)")

Saved: ../../data/processed/schema_csvs/Occupancy.csv (856840 rows, 18 columns)
Saved: ../../data/processed/schema_csvs/Company.csv (234456 rows, 69 columns)
Saved: ../../data/processed/schema_csvs/Succession.csv (36073 rows, 14 columns)
Saved: ../../data/processed/schema_csvs/Person.csv (1121712 rows, 72 columns)
Saved: ../../data/processed/schema_csvs/Ownership.csv (227029 rows, 26 columns)
Saved: ../../data/processed/schema_csvs/LegalEntity.csv (86259 rows, 47 columns)
Saved: ../../data/processed/schema_csvs/Vessel.csv (8913 rows, 34 columns)
Saved: ../../data/processed/schema_csvs/Family.csv (81435 rows, 18 columns)
Saved: ../../data/processed/schema_csvs/Organization.csv (101953 rows, 59 columns)
Saved: ../../data/processed/schema_csvs/Address.csv (30858 rows, 22 columns)
Saved: ../../data/processed/schema_csvs/Directorship.csv (130838 rows, 19 columns)
Saved: ../../data/processed/schema_csvs/Airplane.csv (342 rows, 20 columns)
Saved: ../../data/processed/schema_csvs/Position.csv 

What happened above - 
JSON → flatten → group by schema → save separate CSVs
So, instead of messy table, you get separate table (csv files mentioned below) based on schema

Person.csv

Company.csv

Occupancy.csv

Organization.csv

In [ ]:
# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------
file_path = Path("../../data/processed/schema_csvs/Person.csv")

df = pd.read_csv(file_path)

# -------------------------------------------------------------------
# BASIC OVERVIEW
# -------------------------------------------------------------------
print("Shape (rows, columns):", df.shape)
print("\nColumns:\n", df.columns.tolist())

# -------------------------------------------------------------------
# SAMPLE DATA
# -------------------------------------------------------------------
print("\nFirst 5 rows:")
print(df.head())

# -------------------------------------------------------------------
# DATA TYPES
# -------------------------------------------------------------------
print("\nData types:")
print(df.dtypes)

# -------------------------------------------------------------------
# MISSING VALUES
# -------------------------------------------------------------------
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

# -------------------------------------------------------------------
# NON-NULL COUNTS (useful!)
# -------------------------------------------------------------------
print("\nNon-null counts:")
print(df.count().sort_values(ascending=False))

# -------------------------------------------------------------------
# UNIQUE VALUES (for key columns)
# -------------------------------------------------------------------
print("\nUnique schemas:", df["schema"].unique() if "schema" in df.columns else "N/A")

if "properties.country" in df.columns:
    print("\nUnique countries (sample):")
    print(df["properties.country"].dropna().unique()[:10])

if "properties.topics" in df.columns:
    print("\nUnique topics (sample):")
    print(df["properties.topics"].dropna().unique()[:10])

# -------------------------------------------------------------------
# BASIC STATS (for numeric columns if any)
# -------------------------------------------------------------------
print("\nSummary statistics:")
print(df.describe(include="all"))

# -------------------------------------------------------------------
# COLUMN INSPECTION (VERY IMPORTANT)
# -------------------------------------------------------------------
# pick a few important columns to inspect deeply

cols_to_check = [
    "properties.name",
    "properties.country",
    "properties.birthDate",
    "properties.topics"
]

for col in cols_to_check:
    if col in df.columns:
        print(f"\nSample values for {col}:")
        print(df[col].dropna().head(5))
        
        


/var/folders/s5/p0n_h6hs18s2t0yy41d0kksr0000gn/T/ipykernel_62736/3772928638.py:6: DtypeWarning: Columns (6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,64,65,66,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Shape (rows, columns): (1121712, 72)

Columns:
 ['caption', 'datasets', 'first_seen', 'id', 'last_change', 'last_seen', 'origin', 'properties.abbreviation', 'properties.address', 'properties.addressEntity', 'properties.alias', 'properties.appearance', 'properties.birthCountry', 'properties.birthDate', 'properties.birthPlace', 'properties.citizenship', 'properties.classification', 'properties.country', 'properties.createdAt', 'properties.deathDate', 'properties.description', 'properties.education', 'properties.email', 'properties.ethnicity', 'properties.eyeColor', 'properties.fatherName', 'properties.firstName', 'properties.gender', 'properties.hairColor', 'properties.height', 'properties.icijId', 'properties.idNumber', 'properties.incorporationDate', 'properties.innCode', 'properties.lastName', 'properties.legalForm', 'properties.middleName', 'properties.modifiedAt', 'properties.motherName', 'properties.name', 'properties.nameSuffix', 'properties.nationality', 'properties.notes', 'prop

## Doing some exploratory analysis based on the discussion.
Keeping the Company.csv schema as base, adding other required schemas

In [34]:
#Loading company csv
# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
from pathlib import Path
input_file = Path("../../data/processed")

clean_dir = Path("../../data/processed/cleaned_schemas")

clean_dir.mkdir(parents=True, exist_ok=True)
# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------

import pandas as pd
from pathlib import Path

file_path = Path("../../data/processed/schema_csvs/Company.csv")

df_company = pd.read_csv(file_path)

df.head()


/var/folders/s5/p0n_h6hs18s2t0yy41d0kksr0000gn/T/ipykernel_5675/4020095256.py:20: DtypeWarning: Columns (6,7,9,10,11,12,13,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,60,61,62,63,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_company = pd.read_csv(file_path)


,name,datasets,first_seen,id,last_change,last_seen,origin,properties.abbreviation,address,properties.addressEntity,...,first_seen_time,last_seen_date,last_seen_time,last_change_date,last_change_time,num_datasets,has_sanction_topic,num_topics,num_referents,num_addresses
0,"Myanmar Yatai International Holding Group Co.,...","[us_trade_csl, ext_us_ofac_press_releases, us_...",2025-09-08 14:10:01,NK-223CQDBzp8MRkdJMDiqXn3,2026-02-28 07:55:04,2026-03-02 18:53:02,"[SAM_Exclusions_Public_Extract_V2_26058.CSV, g...",NaN,"[Hpa-An City, Shwe Kokko Village, Myawaddy Tow...","[""addr-ee713d8aee0e4bcfe32761c9d76c0057f43b0f4...",...,14:10:01,2026-03-02,18:53:02,2026-02-28,07:55:04,5,1,2,8,3
1,"Товариство з обмеженою відповідальністю ""Зелін...","[ua_nsdc_sanctions, ext_ru_egrul]",2023-12-13 09:22:01,NK-223yQP6hRaMuiALDCJ6xbY,2026-01-09 03:48:50,2026-03-02 16:22:01,[],NaN,"[Російська Федерація, 115054, м. Москва, вул. ...",NaN,...,09:22:01,2026-03-02,16:22:01,2026-01-09,03:48:50,2,1,1,2,2
2,Open Joint Stock Company “Elektrostal Chemical...,"[mc_fund_freezes, ua_nsdc_sanctions, eu_fsf, e...",2023-12-13 09:22:01,NK-226GXBdQ5p6NjgrTpTQNVW,2026-02-16 22:09:01,2026-03-02 18:25:01,[gpt-5-mini],NaN,"[Electrostal, Karl Marx St., 1, Moscow, Russia...","[""addr-61d6096b4e0706dd5ecb9ad63dcd7e024c55cee...",...,09:22:01,2026-03-02,18:25:01,2026-02-16,22:09:01,7,1,1,7,4
3,"Приватне підприємство ""Магістар-СГ""","[ua_nsdc_sanctions, ext_ua_edr]",2023-04-20 10:50:14,NK-228ZdYZVXaZBSBgVwapnks,2026-01-24 19:25:19,2026-03-02 16:22:01,[],NaN,"[Україна, 79034, Львівська обл., місто Львів, ...",NaN,...,10:50:14,2026-03-02,16:22:01,2026-01-24,19:25:19,2,1,1,2,2
4,"Акціонерне товариство ""Електроагрегат""","[ua_nsdc_sanctions, ext_ru_egrul, ca_dfatd_sem...",2023-04-20 10:50:14,NK-228jBYSTdUSvbZvsKsiHh6,2026-01-09 03:48:50,2026-03-02 18:43:01,[],NaN,"[Російська Федерація, 630015, Новосибірська об...",NaN,...,10:50:14,2026-03-02,18:43:01,2026-01-09,03:48:50,3,1,1,4,3


In [35]:
#cleaning the company dataset
import json

#renaming
df = df_company.copy()

df = df.rename(columns={
    "caption": "name",
    "properties.topics": "topics",
    "properties.address": "address"
})

#convert json-like strings to list

def parse_json_column(x):
    if pd.isna(x):
        return []
    try:
        return json.loads(x)
    except:
        return []

cols_to_parse = ["datasets", "topics", "address", "referents", "origin"]

for col in cols_to_parse:
    if col in df.columns:
        df[col] = df[col].apply(parse_json_column)
        
#covert dates
date_cols = ["first_seen", "last_seen", "last_change"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    
#create time features
df["entity_age_days"] = (df["last_seen"] - df["first_seen"]).dt.days
df["recency_days"] = (pd.Timestamp.today() - df["last_seen"]).dt.days

for col in date_cols:
    df[f"{col}_date"] = df[col].dt.date
    df[f"{col}_time"] = df[col].dt.time


df.head()

,name,datasets,first_seen,id,last_change,last_seen,origin,properties.abbreviation,address,properties.addressEntity,...,schema,target,entity_age_days,recency_days,first_seen_date,first_seen_time,last_seen_date,last_seen_time,last_change_date,last_change_time
0,"Myanmar Yatai International Holding Group Co.,...","[us_trade_csl, ext_us_ofac_press_releases, us_...",2025-09-08 14:10:01,NK-223CQDBzp8MRkdJMDiqXn3,2026-02-28 07:55:04,2026-03-02 18:53:02,"[SAM_Exclusions_Public_Extract_V2_26058.CSV, g...",NaN,"[Hpa-An City, Shwe Kokko Village, Myawaddy Tow...","[""addr-ee713d8aee0e4bcfe32761c9d76c0057f43b0f4...",...,Company,True,175,19,2025-09-08,14:10:01,2026-03-02,18:53:02,2026-02-28,07:55:04
1,"Товариство з обмеженою відповідальністю ""Зелін...","[ua_nsdc_sanctions, ext_ru_egrul]",2023-12-13 09:22:01,NK-223yQP6hRaMuiALDCJ6xbY,2026-01-09 03:48:50,2026-03-02 16:22:01,[],NaN,"[Російська Федерація, 115054, м. Москва, вул. ...",NaN,...,Company,True,810,19,2023-12-13,09:22:01,2026-03-02,16:22:01,2026-01-09,03:48:50
2,Open Joint Stock Company “Elektrostal Chemical...,"[mc_fund_freezes, ua_nsdc_sanctions, eu_fsf, e...",2023-12-13 09:22:01,NK-226GXBdQ5p6NjgrTpTQNVW,2026-02-16 22:09:01,2026-03-02 18:25:01,[gpt-5-mini],NaN,"[Electrostal, Karl Marx St., 1, Moscow, Russia...","[""addr-61d6096b4e0706dd5ecb9ad63dcd7e024c55cee...",...,Company,True,810,19,2023-12-13,09:22:01,2026-03-02,18:25:01,2026-02-16,22:09:01
3,"Приватне підприємство ""Магістар-СГ""","[ua_nsdc_sanctions, ext_ua_edr]",2023-04-20 10:50:14,NK-228ZdYZVXaZBSBgVwapnks,2026-01-24 19:25:19,2026-03-02 16:22:01,[],NaN,"[Україна, 79034, Львівська обл., місто Львів, ...",NaN,...,Company,True,1047,19,2023-04-20,10:50:14,2026-03-02,16:22:01,2026-01-24,19:25:19
4,"Акціонерне товариство ""Електроагрегат""","[ua_nsdc_sanctions, ext_ru_egrul, ca_dfatd_sem...",2023-04-20 10:50:14,NK-228jBYSTdUSvbZvsKsiHh6,2026-01-09 03:48:50,2026-03-02 18:43:01,[],NaN,"[Російська Федерація, 630015, Новосибірська об...",NaN,...,Company,True,1047,19,2023-04-20,10:50:14,2026-03-02,18:43:01,2026-01-09,03:48:50


In [44]:
#dataset count

df["num_datasets"] = df["datasets"].apply(len)

#topics
df["has_sanction_topic"] = df["topics"].apply(lambda x: 1 if "sanction" in x else 0)
df["num_topics"] = df["topics"].apply(len)

#referents (network strength proxy)
df["num_referents"] = df["referents"].apply(len)

#address count
df["num_addresses"] = df["address"].apply(len)



df.head()
for col in df_company.columns:
    print(col)

caption
datasets
first_seen
id
last_change
last_seen
origin
properties.abbreviation
properties.address
properties.addressEntity
properties.alias
properties.amount
properties.bikCode
properties.brightQueryId
properties.brightQueryOrgId
properties.cageCode
properties.cikCode
properties.classification
properties.country
properties.createdAt
properties.description
properties.dissolutionDate
properties.dunsCode
properties.email
properties.giiNumber
properties.icijId
properties.idNumber
properties.imoNumber
properties.incorporationDate
properties.innCode
properties.jurisdiction
properties.keywords
properties.kppCode
properties.legalForm
properties.leiCode
properties.mainCountry
properties.modifiedAt
properties.name
properties.notes
properties.npiCode
properties.ogrnCode
properties.okpoCode
properties.opencorporatesUrl
properties.permId
properties.phone
properties.previousName
properties.program
properties.programId
properties.publisher
properties.registrationNumber
properties.retrievedAt
pro

In [45]:
df.head()

,name,datasets,first_seen,id,last_change,last_seen,origin,properties.abbreviation,address,properties.addressEntity,...,first_seen_time,last_seen_date,last_seen_time,last_change_date,last_change_time,num_datasets,has_sanction_topic,num_topics,num_referents,num_addresses
0,"Myanmar Yatai International Holding Group Co.,...","[us_trade_csl, ext_us_ofac_press_releases, us_...",2025-09-08 14:10:01,NK-223CQDBzp8MRkdJMDiqXn3,2026-02-28 07:55:04,2026-03-02 18:53:02,"[SAM_Exclusions_Public_Extract_V2_26058.CSV, g...",NaN,"[Hpa-An City, Shwe Kokko Village, Myawaddy Tow...","[""addr-ee713d8aee0e4bcfe32761c9d76c0057f43b0f4...",...,14:10:01,2026-03-02,18:53:02,2026-02-28,07:55:04,5,1,2,8,3
1,"Товариство з обмеженою відповідальністю ""Зелін...","[ua_nsdc_sanctions, ext_ru_egrul]",2023-12-13 09:22:01,NK-223yQP6hRaMuiALDCJ6xbY,2026-01-09 03:48:50,2026-03-02 16:22:01,[],NaN,"[Російська Федерація, 115054, м. Москва, вул. ...",NaN,...,09:22:01,2026-03-02,16:22:01,2026-01-09,03:48:50,2,1,1,2,2
2,Open Joint Stock Company “Elektrostal Chemical...,"[mc_fund_freezes, ua_nsdc_sanctions, eu_fsf, e...",2023-12-13 09:22:01,NK-226GXBdQ5p6NjgrTpTQNVW,2026-02-16 22:09:01,2026-03-02 18:25:01,[gpt-5-mini],NaN,"[Electrostal, Karl Marx St., 1, Moscow, Russia...","[""addr-61d6096b4e0706dd5ecb9ad63dcd7e024c55cee...",...,09:22:01,2026-03-02,18:25:01,2026-02-16,22:09:01,7,1,1,7,4
3,"Приватне підприємство ""Магістар-СГ""","[ua_nsdc_sanctions, ext_ua_edr]",2023-04-20 10:50:14,NK-228ZdYZVXaZBSBgVwapnks,2026-01-24 19:25:19,2026-03-02 16:22:01,[],NaN,"[Україна, 79034, Львівська обл., місто Львів, ...",NaN,...,10:50:14,2026-03-02,16:22:01,2026-01-24,19:25:19,2,1,1,2,2
4,"Акціонерне товариство ""Електроагрегат""","[ua_nsdc_sanctions, ext_ru_egrul, ca_dfatd_sem...",2023-04-20 10:50:14,NK-228jBYSTdUSvbZvsKsiHh6,2026-01-09 03:48:50,2026-03-02 18:43:01,[],NaN,"[Російська Федерація, 630015, Новосибірська об...",NaN,...,10:50:14,2026-03-02,18:43:01,2026-01-09,03:48:50,3,1,1,4,3


In [ ]:
#keep useful columns only

keep_cols = [
    "id",
    "name",
    "schema",
    "target",
    "num_datasets",
    "num_topics",
    "has_sanction_topic",
    "num_referents",
    "num_addresses",
    "entity_age_days",
    "recency_days",
    "first_seen_date",
    "last_seen_date",
    "last_change_date",
    
    
    
    
]

df_clean = df[keep_cols]

df_clean.head()

,id,name,schema,target,num_datasets,num_topics,has_sanction_topic,num_referents,num_addresses,entity_age_days,recency_days,first_seen_date,last_seen_date,last_change_date
0,NK-223CQDBzp8MRkdJMDiqXn3,"Myanmar Yatai International Holding Group Co.,...",Company,True,5,2,1,8,3,175,19,2025-09-08,2026-03-02,2026-02-28
1,NK-223yQP6hRaMuiALDCJ6xbY,"Товариство з обмеженою відповідальністю ""Зелін...",Company,True,2,1,1,2,2,810,19,2023-12-13,2026-03-02,2026-01-09
2,NK-226GXBdQ5p6NjgrTpTQNVW,Open Joint Stock Company “Elektrostal Chemical...,Company,True,7,1,1,7,4,810,19,2023-12-13,2026-03-02,2026-02-16
3,NK-228ZdYZVXaZBSBgVwapnks,"Приватне підприємство ""Магістар-СГ""",Company,True,2,1,1,2,2,1047,19,2023-04-20,2026-03-02,2026-01-24
4,NK-228jBYSTdUSvbZvsKsiHh6,"Акціонерне товариство ""Електроагрегат""",Company,True,3,1,1,4,3,1047,19,2023-04-20,2026-03-02,2026-01-09


In [40]:
#saving the cleaned Company.csv

output_path = clean_dir / "company_clean.csv"

df_clean.to_csv(output_path, index=False)

print("Saved at:", output_path)


Saved at: ../../data/processed/cleaned_schemas/company_clean.csv


In [72]:
"""
OpenSanctions — Clean Company Risk Dataset Builder
===================================================
Output: one CSV, one row per company, ready for any downstream use.

Columns produced:
  LABEL
    risk_tier                 : 0 = no flag, 1 = medium risk, 2 = high/direct sanction

  A. GEOGRAPHY
    country                   : best available country code (jurisdiction > mainCountry > country)
    is_high_risk_country      : 1 if country is in high-risk set
    country_missing           : 1 if no country found at all
    jurisdiction_country_mismatch : 1 if incorporation country ≠ operating country

  B. COMPANY TYPE
    legal_form_bucket         : llc_private / joint_stock_public / bank_financial /
                                state_enterprise / foundation_trust / partnership /
                                nonprofit_ngo / other / unknown
    has_sector                : 1 if sector field is populated
    is_holding                : 1 if company name contains holding/group/invest/capital etc.
    is_trading                : 1 if company name contains trading/import/export/commerce

  C. IDENTIFIER RICHNESS
    has_lei / has_duns / has_bic / has_vat / has_tax / has_reg / has_inn / has_ogrn
    identifier_count          : total number of identifiers present (0 = red flag)

  D. ALIAS COMPLEXITY
    alias_count               : number of aliases
    alias_script_diversity    : number of distinct scripts used across aliases
    has_weak_alias            : 1 if weakAlias field is populated
    has_previous_name         : 1 if previousName field is populated

  E. TEMPORAL
    incorporation_year        : year of incorporation
    company_age_years         : CURRENT_YEAR - incorporation_year
    incorporation_missing     : 1 if no incorporation date
    has_dissolution_date      : 1 if dissolution date is present

  F. DIRECTORSHIP (joined from Directorship.csv)
    director_count            : number of unique directors
    active_director_count     : directors with status 'Appointed' / 'ACTIVE'
    resigned_director_count   : directors with status 'Resigned'
    director_role_diversity   : number of unique roles across all directorships
    no_directors_found        : 1 if company has zero directorship records

  G. OWNERSHIP (joined from Ownership.csv)
    owner_count               : number of unique owners
    avg_ownership_pct         : average ownership percentage (where available)
    max_ownership_pct         : maximum ownership percentage (where available)
    has_majority_owner        : 1 if any owner holds > 50%
    ownership_pct_missing     : 1 if no ownership percentages available
    no_owners_found           : 1 if company has zero ownership records
"""

import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
SCHEMA_DIR   = Path("../../data/processed/schema_csvs")
OUTPUT_PATH  = Path("../../data/processed/company_risk_dataset.csv")
CURRENT_YEAR = 2026

HIGH_RISK_COUNTRIES = {
    "ru", "ir", "kp", "sy", "by", "cu", "ve", "mm", "so", "sd",
    "ly", "ye", "zw", "cf", "cd", "iq", "lb", "ni", "af", "ml",
}


# Authoritative risk topics from FollowTheMoney source (topic.py RISKS set)
# https://followthemoney.tech/explorer/types/topic/#data-reference
# These are topics that "imply a counterparty risk in business dealings"
# Note: fin.bank, corp.offshore, corp.shell are deliberately excluded —
# they are descriptive taxonomy only, NOT risk indicators per FTM definition.
FTM_RISK_TOPICS = {
    "corp.disqual",
    "crime", "crime.boss", "crime.fin", "crime.fraud", "crime.terror",
    "crime.theft", "crime.traffick", "crime.war",
    "debarment",
    "export.control", "export.control.linked", "export.risk",
    "invest.risk",
    "mare.detained", "mare.shadow",
    "poi",
    "reg.action", "reg.warn",
    "role.oligarch", "role.pep", "role.rca",
    "sanction", "sanction.counter", "sanction.linked",
    "wanted",
}


LEGAL_FORM_BUCKETS = {
    "llc_private": [
        "llc", "ооо", "sarl", "gmbh", "srl", "lda", "bv", "kft", "sprl",
        "limited liability", "общество с ограниченной", "besloten",
        "société à responsabilité", "private limited", "pvt", "pte",
    ],
    "joint_stock_public": [
        "jsc", "ojsc", "pjsc", "ао ", "пао", " ao ", "акционерное общество",
        "joint stock", "open joint", " sa ", " ag ", "plc", " nv ", "nyrt",
        "public limited", "sociedade anónima",
    ],
    "bank_financial": [
        "bank", "банк", "credit", "кредит", "финансов", "insurance",
        "страхов", "investment fund", "инвестиционный фонд",
    ],
    "state_enterprise": [
        "state", "federal", "unitary", "фгуп", "гуп", "казенное",
        "государственн", "municipal", "муниципальн",
    ],
    "foundation_trust": [
        "foundation", "trust", "stiftung", "fondation", "fundación",
        "charity", "endowment",
    ],
    "partnership": [
        "llp", " lp ", "partnership", "коммандитн", "полное товарищество",
    ],
    "nonprofit_ngo": [
        "ngo", "association", "некоммерческ", "non-profit", "nonprofit",
        "cooperative", "кооператив",
    ],
}

HOLDING_KEYWORDS  = ["holding", "holdings", "group", "invest", "capital",
                      "international", "global", "ventures", "partners"]
TRADING_KEYWORDS  = ["trading", "trade", "import", "export", "commerce",
                      "commodities", "supply", "logistics"]
# ─────────────────────────────────────────────────────────────────────────────

# ═════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def load(filename):
    path = SCHEMA_DIR / filename
    df = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {filename}: {len(df):,} rows")
    return df


def parse_list_field(series):
    """'["a", "b"]' → actual Python list. Returns Series of lists."""
    def _parse(val):
        if pd.isna(val):
            return []
        try:
            result = ast.literal_eval(str(val))
            return result if isinstance(result, list) else [result]
        except Exception:
            return [str(val)]
    return series.apply(_parse)


def first_value(series):
    """First item from a stringified list column, lowercased. NaN if empty."""
    return parse_list_field(series).apply(
        lambda x: x[0].strip().lower() if x else np.nan
    )


def clean_id(series):
    """'["abc123"]' → 'abc123'"""
    return parse_list_field(series).apply(
        lambda x: x[0].strip() if x else np.nan
    )


def count_list_field(series):
    """Number of items in a stringified list column."""
    return parse_list_field(series).apply(len)


# ═════════════════════════════════════════════════════════════════════════════
# LOAD
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Loading files ───────────────────────────────────────────")
company      = load("Company.csv")
directorship = load("Directorship.csv")
ownership    = load("Ownership.csv")

# ═════════════════════════════════════════════════════════════════════════════
# LOAD
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Loading files ───────────────────────────────────────────")
company      = load("Company.csv")
directorship = load("Directorship.csv")
ownership    = load("Ownership.csv")


# ═════════════════════════════════════════════════════════════════════════════
# LABEL
# ═════════════════════════════════════════════════════════════════════════════
print("\n── Building label ──────────────────────────────────────────")

def assign_risk_tier(topics_val):
    if pd.isna(topics_val) or str(topics_val).strip() in ("", "[]", "nan"):
        return 0

    try:
        topics_list = ast.literal_eval(str(topics_val))
    except Exception:
        topics_list = [str(topics_val)]

    topics_set = {t.strip().lower() for t in topics_list}

    # Tier 2 — directly sanctioned:
    # 'sanction' present but NOT via 'sanction.linked' or 'sanction.counter'
    # (those are risk-adjacent, not direct listing)
    if "sanction" in topics_set and not {"sanction.linked", "sanction.counter"} & topics_set:
        return 2

    # Tier 1 — any other official FTM risk topic
    # Source: followthemoney/types/topic.py RISKS set
    if topics_set & FTM_RISK_TOPICS:
        return 1

    # Tier 0 — descriptive only (fin.bank, corp.offshore, corp.shell, gov.*, etc.)
    return 0

company["risk_tier"] = company["properties.topics"].apply(assign_risk_tier)
counts = company["risk_tier"].value_counts().sort_index()
print(f"  Tier 0 — no flag:     {counts.get(0, 0):,}")
print(f"  Tier 1 — medium risk: {counts.get(1, 0):,}")
print(f"  Tier 2 — high risk:   {counts.get(2, 0):,}")



# ═════════════════════════════════════════════════════════════════════════════
# E. TEMPORAL
# ═════════════════════════════════════════════════════════════════════════════
print("\n── E. Temporal ─────────────────────────────────────────────")

#feat = pd.DataFrame(index=company.index)

incorp_raw = first_value(company["properties.incorporationDate"])
incorp_year = pd.to_datetime(incorp_raw, errors="coerce").dt.year

feat["incorporation_year"]    = incorp_year
feat["company_age_years"]     = CURRENT_YEAR - incorp_year
feat["incorporation_missing"] = incorp_year.isna().astype(int)
feat["has_dissolution_date"]  = company["properties.dissolutionDate"].notna().astype(int)

# ═════════════════════════════════════════════════════════════════════════════
# F. DIRECTORSHIP (aggregated)
# ═════════════════════════════════════════════════════════════════════════════
print("\n── F. Directorship ─────────────────────────────────────────")

directorship["org_id"]    = clean_id(directorship["properties.organization"])
directorship["dir_id"]    = clean_id(directorship["properties.director"])
directorship["role_raw"]  = directorship["properties.role"].fillna("[]")
directorship["status_raw"] = directorship["properties.status"].fillna("").str.lower()

directorship["is_active"]   = directorship["status_raw"].str.contains("appointed|active", na=False).astype(int)
directorship["is_resigned"] = directorship["status_raw"].str.contains("resigned", na=False).astype(int)

def count_unique_roles(role_val):
    try:
        roles = ast.literal_eval(str(role_val))
        return len(set(roles)) if isinstance(roles, list) else 1
    except Exception:
        return 1

directorship["role_count"] = directorship["role_raw"].apply(count_unique_roles)

dir_agg = (
    directorship
    .groupby("org_id")
    .agg(
        director_count          =("dir_id",     "nunique"),
        active_director_count   =("is_active",  "sum"),
        resigned_director_count =("is_resigned", "sum"),
        director_role_diversity =("role_count", "sum"),
    )
    .reset_index()
    .rename(columns={"org_id": "id"})
)

feat = feat.merge(dir_agg, on="id", how="left")
for col in ["director_count", "active_director_count",
            "resigned_director_count", "director_role_diversity"]:
    feat[col] = feat[col].fillna(0).astype(int)

feat["no_directors_found"] = (feat["director_count"] == 0).astype(int)



── Loading files ───────────────────────────────────────────
  Loaded Company.csv: 234,456 rows
  Loaded Directorship.csv: 130,838 rows
  Loaded Ownership.csv: 227,029 rows

── Loading files ───────────────────────────────────────────
  Loaded Company.csv: 234,456 rows
  Loaded Directorship.csv: 130,838 rows
  Loaded Ownership.csv: 227,029 rows

── Building label ──────────────────────────────────────────
  Tier 0 — no flag:     126,612
  Tier 1 — medium risk: 93,875
  Tier 2 — high risk:   13,969

── E. Temporal ─────────────────────────────────────────────

── F. Directorship ─────────────────────────────────────────


KeyError: 'id'

In [65]:
company[["id", "risk_tier", "target"]].head(10)

,id,risk_tier,target
0,NK-223CQDBzp8MRkdJMDiqXn3,2,True
1,NK-223yQP6hRaMuiALDCJ6xbY,2,True
2,NK-226GXBdQ5p6NjgrTpTQNVW,2,True
3,NK-228ZdYZVXaZBSBgVwapnks,2,True
4,NK-228jBYSTdUSvbZvsKsiHh6,2,True
5,NK-22HqVo5sJ5jpegvbuw3gZX,0,False
6,NK-22MHXvQQufBgTjUWgUbWb8,2,True
7,NK-22T6dTsc6umYgEA33vME3t,2,True
8,NK-22WmG9GWt6yirMLDHUEXMq,1,True
9,NK-22YnEcmYTBXeCBBeZU5wCN,2,True


In [71]:
# ═════════════════════════════════════════════════════════════════════════════
# F. DIRECTORSHIP (aggregated)
# ═════════════════════════════════════════════════════════════════════════════
print("\n── F. Directorship ─────────────────────────────────────────")

directorship["org_id"]    = clean_id(directorship["properties.organization"])
directorship["dir_id"]    = clean_id(directorship["properties.director"])
directorship["role_raw"]  = directorship["properties.role"].fillna("[]")
directorship["status_raw"] = directorship["properties.status"].fillna("").str.lower()

directorship["is_active"]   = directorship["status_raw"].str.contains("appointed|active", na=False).astype(int)
directorship["is_resigned"] = directorship["status_raw"].str.contains("resigned", na=False).astype(int)

def count_unique_roles(role_val):
    try:
        roles = ast.literal_eval(str(role_val))
        return len(set(roles)) if isinstance(roles, list) else 1
    except Exception:
        return 1

directorship["role_count"] = directorship["role_raw"].apply(count_unique_roles)

dir_agg = (
    directorship
    .groupby("org_id")
    .agg(
        director_count          =("dir_id",     "nunique"),
        active_director_count   =("is_active",  "sum"),
        resigned_director_count =("is_resigned", "sum"),
        director_role_diversity =("role_count", "sum"),
    )
    .reset_index()
    .rename(columns={"org_id": "id"})
)

feat = feat.merge(dir_agg, on="id", how="left")
for col in ["director_count", "active_director_count",
            "resigned_director_count", "director_role_diversity"]:
    feat[col] = feat[col].fillna(0).astype(int)

feat["no_directors_found"] = (feat["director_count"] == 0).astype(int)


── F. Directorship ─────────────────────────────────────────


KeyError: 'id'